In [ ]:
UNI_RANDOM_SEED = 2024

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

import pdb
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False

from cudaext.ops.Rotated_IoU.oriented_iou_loss import cal_iou_3d, assign_target_3d

from pytorch3d.ops import sample_points_from_meshes, laplacian
from pytorch3d.loss import mesh_laplacian_smoothing
from pytorch3d.structures import Meshes, join_meshes_as_batch
from pytorch3d.utils import ico_sphere
from pytorch3d.transforms import Scale
from pytorch3d.vis.plotly_vis import AxisArgs, plot_batch_individually, plot_scene

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from data_tools import adv_dataset, kitti_carla_dataset
from eval_utils import eval_utils
from loss_utils import mesh_objectwise_loss


EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Kitti Attack Test-------------------------')

laplacian_weights = 1.0
learning_rate = 0.001


In [ ]:
def roipooling_grad_mapping(pooled_features_grad, batch_point_features, pooled_pts_idx):
    
    batch_size = batch_point_features.size(0)
    npoint = batch_point_features.size(1)
    feature_size = batch_point_features.size(2)
    
    batch_point_features_grad = torch.zeros_like(batch_point_features)
    xyz_features_grad = batch_point_features.new_zeros((batch_size, npoint, 3))
    
    for batch_mask in range(0, batch_size):
        pts_idx_expanded = pooled_pts_idx[batch_mask].view(-1).long().unsqueeze(1).expand(-1, feature_size)
        xyz_pooled_features_grad_viewed = pooled_features_grad[:, :, :3].view(-1, 3)
        pooled_features_grad_viewed = pooled_features_grad[:, :, 3:].view(-1, feature_size)

        batch_point_features_grad[batch_mask].scatter_add_(0, pts_idx_expanded, pooled_features_grad_viewed)
        xyz_features_grad[batch_mask].scatter_add_(0, pts_idx_expanded[:, :3], xyz_pooled_features_grad_viewed)
    
    return xyz_features_grad, batch_point_features_grad



In [ ]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

In [ ]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

kitti_adv_dataset = adv_dataset(test_set,
                                surrogate_model=None)
optimizer = optim.Adam([kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert()], 
                       lr=learning_rate)

In [ ]:
kitti_adv_dataset.enable_adversarial_patch(True)
eval_utils.eval_one_epoch(
        cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

Car AP@0.70, 0.70, 0.70:
bbox AP:81.2036, 74.7322, 67.8265
bev  AP:79.6103, 72.2210, 66.4427
3d   AP:67.8708, 60.0782, 54.4692
aos  AP:80.31, 73.21, 66.26
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:82.6373, 76.3993, 70.4407
bev  AP:79.9304, 73.0078, 67.2479
3d   AP:68.0128, 58.7108, 52.8206
aos  AP:81.68, 74.69, 68.51
Car AP@0.70, 0.50, 0.50:
bbox AP:81.2036, 74.7322, 67.8265
bev  AP:81.9749, 75.8437, 73.9331
3d   AP:81.8360, 75.6438, 73.7331
aos  AP:80.31, 73.21, 66.26
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:82.6373, 76.3993, 70.4407
bev  AP:83.4352, 79.0004, 73.1383
3d   AP:83.2994, 78.7224, 72.8745
aos  AP:81.68, 74.69, 68.51

In [ ]:
criterion = mesh_objectwise_loss(freezed_iou = True, 
                                 normalized = False, 
                                 verbose=False)
def evaluate_one_epoch_attack(enable_adv, update, visualize, verbose_epoch: int = 100):
    mesh_loss_scaler = []
    regular_loss_scaler = []
    kitti_adv_dataset.enable_adversarial_patch(enable_adv)
    
    for i, batch_dict in tqdm(enumerate(kitti_adv_dataset)):
        load_data_to_gpu(batch_dict)

        model.eval()
        model.zero_grad()
        pred_dicts, _ = model(batch_dict)
        point_headbox_ret_dict = point_headbox.forward_ret_dict
        pointrcnn_head_ret_dict = pointrcnn_head.forward_ret_dict
        
        if batch_dict['gt_boxes'].size(1) == 0:
            logger.info(f"no vehicles found in batch \t{i}")
            continue
        
        mesh_loss = criterion(batch_dict = point_headbox_ret_dict, 
                                point_coords = batch_dict["point_coords"][:, 1:4].squeeze(dim=0),
                                gt_boxes = batch_dict["gt_boxes"], 
                                target_class = 1,
                                ret_part_loss = False)
        regular_loss = kitti_adv_dataset.universal_adv_patch.get_laplacian_loss()
        total_loss = mesh_loss + laplacian_weights * regular_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        
        mesh_loss_scaler.append(mesh_loss.item())
        regular_loss_scaler.append(regular_loss.item())
            
        if verbose_epoch > 0 and i % verbose_epoch == 0:
            logger.info(f"average mesh loss: \t{np.array(mesh_loss_scaler).mean()}")
            logger.info(f"average regular loss: \t{np.array(regular_loss_scaler).mean()}")
            # logger.info(f"deformed verts of mesh: \t{kitti_adv_dataset.universal_adv_patch.get_mesh_deform_vert()}")
            
            if visualize:
                fig = plot_scene({
                    "original": {
                        "mesh_1": kitti_adv_dataset.universal_adv_patch.get_basic_mesh()
                    },
                    "adversarial": {
                        "mesh_1": kitti_adv_dataset.universal_adv_patch.get_deformed_mesh()
                    },
                    
                }, ncols=2)
                fig.update_layout(height=400, width=800)
                fig.show()
                
                V.draw_scenes(
                    points=batch_dict['points'][:, 1:], ref_boxes=pred_dicts[0]['pred_boxes'].detach(),
                    ref_scores=pred_dicts[0]['pred_scores'].detach(), ref_labels=pred_dicts[0]['pred_labels'].detach(), gt_boxes=batch_dict['gt_boxes'][0]
                )
            
        if update:
            optimizer.step()
            
    return mesh_loss_scaler, regular_loss_scaler

mesh_loss_scaler, regular_loss_scaler = evaluate_one_epoch_attack(enable_adv = True, 
                                          update = True, 
                                          visualize = True, 
                                          verbose_epoch=-1)


In [ ]:
plt.plot(np.arange(mesh_loss_scaler.__len__()), mesh_loss_scaler, label='Mesh loss')

# # 绘制第二条曲线
# plt.plot(x_values, y2_values, label='Curve 2')

# 添加图例
plt.legend()

# 添加标题和坐标轴标签
plt.title('Mesh loss')
plt.xlabel('X Axis')
plt.ylabel('Y Axis')

# 显示图形
plt.show()

In [ ]:
import pdb
pdb.set_trace()
kitti_adv_dataset.enable_adversarial_patch(True)
eval_utils.eval_one_epoch(
        cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

Car AP@0.70, 0.70, 0.70:
bbox AP:81.2036, 74.7322, 67.8265
bev  AP:79.6103, 72.2210, 66.4427
3d   AP:67.8708, 60.0782, 54.4692
aos  AP:80.31, 73.21, 66.26
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:82.6373, 76.3993, 70.4407
bev  AP:79.9304, 73.0078, 67.2479
3d   AP:68.0128, 58.7108, 52.8206
aos  AP:81.68, 74.69, 68.51
Car AP@0.70, 0.50, 0.50:
bbox AP:81.2036, 74.7322, 67.8265
bev  AP:81.9749, 75.8437, 73.9331
3d   AP:81.8360, 75.6438, 73.7331
aos  AP:80.31, 73.21, 66.26
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:82.6373, 76.3993, 70.4407
bev  AP:83.4352, 79.0004, 73.1383
3d   AP:83.2994, 78.7224, 72.8745
aos  AP:81.68, 74.69, 68.51

Car AP@0.70, 0.70, 0.70:
bbox AP:80.5489, 74.2782, 67.7490
bev  AP:78.7853, 71.7193, 66.1230
3d   AP:67.7154, 59.9228, 54.3259
aos  AP:79.43, 72.43, 65.77
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:80.9223, 74.6884, 70.2471
bev  AP:78.8351, 71.1878, 66.9471
3d   AP:67.9778, 58.3919, 52.8153
aos  AP:79.78, 72.73, 67.85
Car AP@0.70, 0.50, 0.50:
bbox AP:80.5489, 74.2782, 67.7490
bev  AP:81.0613, 75.5435, 73.8589
3d   AP:80.8861, 75.3491, 73.6195
aos  AP:79.43, 72.43, 65.77
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:80.9223, 74.6884, 70.2471
bev  AP:81.4775, 78.6534, 73.0656
3d   AP:81.3126, 77.0607, 72.7528
aos  AP:79.78, 72.73, 67.85

In [ ]:
import pdb
pdb.set_trace()
kitti_adv_dataset.enable_adversarial_patch(False)
eval_utils.eval_one_epoch(
        cfg, None, model, kitti_adv_dataset, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

2024-01-12 15:47:35,915   INFO  
Car AP@0.70, 0.70, 0.70:
bbox AP:98.0106, 90.4862, 90.3078
bev  AP:90.3662, 88.9447, 88.5946
3d   AP:89.2901, 79.2153, 78.7551
aos  AP:97.96, 90.39, 90.13
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:99.0887, 94.0299, 93.9594
bev  AP:95.9050, 90.3332, 90.2202
3d   AP:92.1830, 81.4108, 80.8872
aos  AP:99.06, 93.90, 93.74
Car AP@0.70, 0.50, 0.50:
bbox AP:98.0106, 90.4862, 90.3078
bev  AP:98.3867, 90.6892, 90.6100
3d   AP:98.2945, 90.6742, 90.5853
aos  AP:97.96, 90.39, 90.13
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:99.0887, 94.0299, 93.9594
bev  AP:99.3582, 96.7091, 96.7086
3d   AP:99.3153, 96.6458, 96.5752
aos  AP:99.06, 93.90, 93.74

Car AP@0.70, 0.70, 0.70:
bbox AP:88.8749, 86.7349, 80.0941
bev  AP:87.4082, 79.2386, 78.7990
3d   AP:75.4462, 66.8881, 65.7855
aos  AP:87.93, 84.61, 77.75
Car AP_R40@0.70, 0.70, 0.70:
bbox AP:91.9898, 86.8892, 84.6147
bev  AP:90.3528, 83.3402, 81.1508
3d   AP:78.5963, 69.1575, 65.2876
aos  AP:90.93, 84.74, 81.73
Car AP@0.70, 0.50, 0.50:
bbox AP:88.8749, 86.7349, 80.0941
bev  AP:89.5853, 88.6711, 88.1263
3d   AP:89.5065, 88.4047, 87.9617
aos  AP:87.93, 84.61, 77.75
Car AP_R40@0.70, 0.50, 0.50:
bbox AP:91.9898, 86.8892, 84.6147
bev  AP:94.5442, 90.2026, 87.9913
3d   AP:92.6863, 90.0026, 87.7796
aos  AP:90.93, 84.74, 81.73